# SIADS 593 WN26 Milestone I: Sustainable Water Quality

**Authors:**

Sungmin Kim  
Randy Best  
Kyle Rodriguez

### **Introduction**

This notebook will invlove standardizing data, joining population density using a nearest-neighbor approach, validating the join with distance checks, constructing river mouth/junction proximity features, and exporting a dataset
ready to use for plotting visualizations.  

Second, the objective is to enrich water-quality observations with environmental and spatial context (population density proximity and river mouth/junction proximity) to support downstream exploratory analysis. Each water-quality sample will be matched
to the nearest 1 km population grid cell from the same year—and the distance to that grid cell is imputed to validate spatial alignment.  

Finally, river mouth and junction features are defined with a chosen, 10 km proximity threshold.

### **Load Data**

In [1]:
# Data manipulation and analysis
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree
from scipy.spatial import cKDTree

# Modules
from utils import pop_density, world_pop

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

%load_ext watermark
%watermark -v
%watermark --iversions

Python implementation: CPython
Python version       : 3.12.12
IPython version      : 9.4.0

numpy  : 2.3.2
sklearn: 1.7.1
pandas : 2.3.1
scipy  : 1.16.1



#### ...Now we can join the population density variable to the dataset.

#### Secondary Dataset 2.2 - Population Density

In [2]:
# Follow the yellow-brick-road
path = 'data/Population_density'
filenames = pop_density.pop_density()
filenames
# Source - https://stackoverflow.com/q/20906474
# Posted by jonas, modified by community. See post 'Timeline' for change history
# Retrieved 2026-01-30, License - CC BY-SA 4.0

['data/Population_density/zaf_pd_2012_1km_UNadj_ASCII_XYZ.zip',
 'data/Population_density/zaf_pd_2014_1km_UNadj_ASCII_XYZ.zip',
 'data/Population_density/zaf_pd_2013_1km_UNadj_ASCII_XYZ.zip',
 'data/Population_density/zaf_pd_2011_1km_UNadj_ASCII_XYZ.zip',
 'data/Population_density/zaf_pd_2015_1km_UNadj_ASCII_XYZ.zip']

- Create a list of pandas dataframes using population density
- Each dataframe has 3 columns X (longitude), Y (latitude) and Z (population density in [units])

In [3]:
all_dfs = [world_pop.world_pop(filename) for filename in filenames]

# Combine all countries & years into one DataFrame
population_density_all = pd.concat(all_dfs, ignore_index=True)

# Sort for consistency
population_density_all = population_density_all.sort_values(
    by=["country", "year", "latitude", "longitude", "population_density"]
).reset_index(drop=True)

# Quick check before joining; you'll notice the countries are all "Unknown"
population_density_all.head()

Population density from pd ['data/Population', 'density/zaf', 'pd', '2012', '1km', 'UNadj', 'ASCII', 'XYZ.zip']
Population density from pd ['data/Population', 'density/zaf', 'pd', '2014', '1km', 'UNadj', 'ASCII', 'XYZ.zip']
Population density from pd ['data/Population', 'density/zaf', 'pd', '2013', '1km', 'UNadj', 'ASCII', 'XYZ.zip']
Population density from pd ['data/Population', 'density/zaf', 'pd', '2011', '1km', 'UNadj', 'ASCII', 'XYZ.zip']
Population density from pd ['data/Population', 'density/zaf', 'pd', '2015', '1km', 'UNadj', 'ASCII', 'XYZ.zip']


,country,year,longitude,latitude,population_density
0,Unknown,2011,37.794583,-46.979583,0.0
1,Unknown,2011,37.802917,-46.979583,0.0
2,Unknown,2011,37.811250,-46.979583,0.0
3,Unknown,2011,37.819583,-46.979583,0.0
4,Unknown,2011,37.827917,-46.979583,0.0


### **Join** 

For each water-quality sampling point, associate the nearest population-density grid value (pop_density_nn) from its sample year accordingly.  
Also impute the distance of the sample site to the population density cell (distance_km_to_pd_cell) to check our sanity that the join is spatially reasonable.

In [4]:
# --- 1) Copies + normalize column names ---
wq = pd.read_pickle('data/i_need_pop_density.pkl')
pd_all = population_density_all.copy()

wq.columns = wq.columns.str.strip().str.lower()
pd_all.columns = pd_all.columns.str.strip().str.lower()

# --- 2) Ensure numeric types ---
for col in ["longitude", "latitude"]:
    wq[col] = pd.to_numeric(wq[col], errors="coerce")
    pd_all[col] = pd.to_numeric(pd_all[col], errors="coerce")

pd_all["population_density"] = pd.to_numeric(pd_all["population_density"], errors="coerce")
pd_all["year"] = pd.to_numeric(pd_all["year"], errors="coerce")

# --- 3) Extract sample year ---
wq["sample_year"] = pd.to_datetime(wq["sample date"], errors="coerce").dt.year

# --- 4) Initialize output columns ---
wq["pd_year"] = np.nan
wq["pop_density_nn"] = np.nan
wq["distance_km_to_pd_cell"] = np.nan

wq.head()

,province,country,latitude,longitude,sample date,nir,green,swir16,swir22,ndmi,mndwi,pet,total alkalinity,electrical conductance,dissolved reactive phosphorus,month,sample_year,pd_year,pop_density_nn,distance_km_to_pd_cell
1,Mpumalanga,South Africa,-26.861111,28.884722,2011-01-03,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134,124.1,74.720,162.9,163.0,2011-01-31,2011,NaN,NaN,NaN
2,Gauteng,South Africa,-26.450000,28.085833,2011-01-03,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805,127.5,89.254,573.0,80.0,2011-01-31,2011,NaN,NaN,NaN
3,Free State,South Africa,-27.671111,27.236944,2011-01-03,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416,129.7,82.000,203.6,101.0,2011-01-31,2011,NaN,NaN,NaN
4,Free State,South Africa,-27.356667,27.286389,2011-01-03,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683,129.2,56.100,145.1,151.0,2011-01-31,2011,NaN,NaN,NaN
5,Free State,South Africa,-27.010111,26.698083,2011-01-04,12433.5,10433.5,9579.5,8531.5,0.129651,0.042672,98.6,82.200,289.8,192.0,2011-01-31,2011,NaN,NaN,NaN


#### After applying some data cleaning, paying close attention to dtypes and getting rid of spaces or capatilized letters in the column names, we add three empty columns to fill in.

- "pd_year" -> the year of the population density estimate
- "pop_density_nn" -> the population density in the nearest square kilometer to the sample site
- "distance_km_to_pd_cell" -> distance in kilometers from the sample site to the population density coordinates in ```population_density_all``` (should be less than 1 km)

In [5]:
# --- 5) Helper: approx distance in km ---
def approx_deg_to_km(dlon, dlat, lat):
    '''
    input
    -----
    dlon: distance between 2 longitudinal lines
    dlat: distance between 2 latitudinal lines
    lat: distance between 
    return
    ------
    float: distance between 2 points in km
    '''
    km_lat = dlat * 111.32
    km_lon = dlon * 111.11 * np.cos(np.deg2rad(lat))
    return np.sqrt(km_lat**2 + km_lon**2)

# --- 6) Filter PD to clean rows only ---
pd_all_clean = pd_all.dropna(subset=["year", "longitude", "latitude", "population_density"]).copy()
pd_all_clean["year"] = pd_all_clean["year"].astype(int)
print("There are ",len(pd_all_clean),"population densities")
print(len(pd_all_clean[pd_all_clean['country'].isna()==True]),"of them are null")

There are  8157545 population densities
0 of them are null


In [27]:
# --- 7) Build KDTree per year once ---
trees = {}
pd_by_year = {}

# Group population density grid cells by year and reset index so positional indexing line up with KDTree output
# Use coordinates for nearest-neighbor matching and build KDTree for fast nearest point search
for yr, group in pd_all_clean.groupby("year"):
    group = group.reset_index(drop=True)
    coords = group[["longitude", "latitude"]].to_numpy()
    trees[yr] = cKDTree(coords)
    # Save the year-specific dataframe in pd_by_year dictionary
    pd_by_year[yr] = group

print('last year from the loop, and its group:\n',yr, group)
print("Every year is a key, and every group is the dataframe of locations with their population densities recorded in that year")
print('FYI: trees',trees.values())

last year from the loop, and its group:
 2015          country  year  longitude   latitude  population_density
0        Unknown  2015  37.794583 -46.979583            0.000000
1        Unknown  2015  37.802917 -46.979583            0.000000
2        Unknown  2015  37.811250 -46.979583            0.000000
3        Unknown  2015  37.819583 -46.979583            0.000000
4        Unknown  2015  37.827917 -46.979583            0.000000
...          ...   ...        ...        ...                 ...
1631504  Unknown  2015  29.627917 -22.129583           62.707832
1631505  Unknown  2015  29.636250 -22.129583           26.883219
1631506  Unknown  2015  29.644583 -22.129583           34.937401
1631507  Unknown  2015  29.652917 -22.129583          131.900742
1631508  Unknown  2015  29.661250 -22.129583          100.747963

[1631509 rows x 5 columns]
Every year is a key, and every group is the dataframe of locations with their population densities recorded in that year
FYI: trees dict_values([<

In [40]:
# --- 8) Match WQ points year-by-year ---
# Keep only sampling rows with valid year + coordinates
valid_wq = wq.dropna(subset=["sample_year", "longitude", "latitude"]).copy()
valid_wq["sample_year"] = valid_wq["sample_year"].astype(int)

# Loop through each year that appears in the sampling data
# If population data doesn't exist for this year, skip
for yr in sorted(valid_wq["sample_year"].unique()):
    if yr not in trees:
        print(yr,'not in trees')
        continue

    # Indices for sampling points this year
    # Coordinates for sampling points this year
    idx_rows = valid_wq.index[valid_wq["sample_year"] == yr]
    query_pts = valid_wq.loc[idx_rows, ["longitude", "latitude"]].to_numpy()

    # KDTree nearest-neighbor query:
    # dist -> distance in degrees (not km)
    # idx -> index of nearest population grid cell row (within pd_by_year[yr])
    # Retrieve the matched population density rows
    dist, idx = trees[yr].query(query_pts, k=1)
    matched = pd_by_year[yr].iloc[idx].reset_index(drop=True)

    # Store matched year + population density back into the original wq dataframe
    wq.loc[idx_rows, "pd_year"] = yr
    wq.loc[idx_rows, "pop_density_nn"] = matched["population_density"].to_numpy()

    # Compute km distance between sampling point and matched grid cell centroid
    dlon = valid_wq.loc[idx_rows, "longitude"].to_numpy() - matched["longitude"].to_numpy()
    dlat = valid_wq.loc[idx_rows, "latitude"].to_numpy() - matched["latitude"].to_numpy()
    lat  = valid_wq.loc[idx_rows, "latitude"].to_numpy()

    wq.loc[idx_rows, "distance_km_to_pd_cell"] = approx_deg_to_km(dlon, dlat, lat)

print('Population Density has been added for each sample\nIt is the estimated population per square km\n',round(wq['pop_density_nn'].describe(), None).astype(int))
wq.head()

Population Density has been added for each sample
It is the estimated population per square km
 count    9171
mean      101
std       453
min         0
25%         2
50%        10
75%        49
max      6180
Name: pop_density_nn, dtype: int64


,province,country,latitude,longitude,sample date,nir,green,swir16,swir22,ndmi,mndwi,pet,total alkalinity,electrical conductance,dissolved reactive phosphorus,month,sample_year,pd_year,pop_density_nn,distance_km_to_pd_cell
1,Mpumalanga,South Africa,-26.861111,28.884722,2011-01-03,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134,124.1,74.720,162.9,163.0,2011-01-31,2011,2011.0,5.049022,0.251645
2,Gauteng,South Africa,-26.450000,28.085833,2011-01-03,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805,127.5,89.254,573.0,80.0,2011-01-31,2011,2011.0,23.239988,0.419541
3,Free State,South Africa,-27.671111,27.236944,2011-01-03,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416,129.7,82.000,203.6,101.0,2011-01-31,2011,2011.0,687.465759,0.070024
4,Free State,South Africa,-27.356667,27.286389,2011-01-03,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683,129.2,56.100,145.1,151.0,2011-01-31,2011,2011.0,6.092811,0.232396
5,Free State,South Africa,-27.010111,26.698083,2011-01-04,12433.5,10433.5,9579.5,8531.5,0.129651,0.042672,98.6,82.200,289.8,192.0,2011-01-31,2011,2011.0,77.849716,0.466438


In [41]:
# --- 9) Build FINAL joined dataset: all original columns + new features ---
original_cols = pd.read_pickle('data/i_need_pop_density.pkl')
original_cols.columns = original_cols.columns.str.strip().str.lower()
original_cols = original_cols.columns.tolist()

new_cols = ["sample_year", "pop_density_nn", "distance_km_to_pd_cell"]

# Keep original order + add new cols at the end (only if they exist)
final_cols = original_cols + [c for c in new_cols if c in wq.columns]
wq_joined = wq[final_cols].copy()

# --- 10) Take a peak ---
wq_joined.head()

,province,country,latitude,longitude,sample date,nir,green,swir16,swir22,ndmi,mndwi,pet,total alkalinity,electrical conductance,dissolved reactive phosphorus,month,sample_year,pop_density_nn,distance_km_to_pd_cell
1,Mpumalanga,South Africa,-26.861111,28.884722,2011-01-03,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134,124.1,74.720,162.9,163.0,2011-01-31,2011,5.049022,0.251645
2,Gauteng,South Africa,-26.450000,28.085833,2011-01-03,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805,127.5,89.254,573.0,80.0,2011-01-31,2011,23.239988,0.419541
3,Free State,South Africa,-27.671111,27.236944,2011-01-03,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416,129.7,82.000,203.6,101.0,2011-01-31,2011,687.465759,0.070024
4,Free State,South Africa,-27.356667,27.286389,2011-01-03,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683,129.2,56.100,145.1,151.0,2011-01-31,2011,6.092811,0.232396
5,Free State,South Africa,-27.010111,26.698083,2011-01-04,12433.5,10433.5,9579.5,8531.5,0.129651,0.042672,98.6,82.200,289.8,192.0,2011-01-31,2011,77.849716,0.466438


In [45]:
wq_joined["distance_km_to_pd_cell"].agg(
    ["min", "max", "mean", "median"]
)

min       0.025901
max       0.783485
mean      0.347578
median    0.364569
Name: distance_km_to_pd_cell, dtype: float64

**How to interpret this**

You’re matching water-quality points to 1 km population-density grid cells.

Your results:
	•	Mean distance ≈ 0.34 km (340 meters)
	•	Median ≈ 0.36 km
	•	Max ≈ 0.61 km

For a 1 km grid, this is exactly what we want to see.

Why?

	•	In a 1 km × 1 km cell, the furthest possible distance from the center is ~0.71 km.
	•	Your max (0.61 km) is below that theoretical worst case.
	•	A mean around 0.35 km means most matches are happening well within the same grid cell or very close neighbors.

This tells us:

	•	Spatial alignment is tight
	•	Nearest-neighbor matching worked correctly
	•	No red flags like multi-km mismatches

## Lastly Join River Names and Junction Coordinates

```rivers``` : large natural streams of water emptying into an ocean, river or other body of water.  
```river_mouths``` : where a river ends in a body of water, such as a lake, sea or ocean.  
```river_junctions``` : where a river ends by intersecting, or running into another river.

**NOTE: River mouth and river junction cannot be used interchangeably.**

In [46]:
rivers = pd.read_pickle('data/rivers.pkl')
river_mouths = pd.read_pickle('data/river_mouths.pkl')

# distinguish river mouths from junctions
river_junctions = rivers[~rivers['river'].isin(river_mouths['river'])]
river_junctions.head()

,river,drainagebasin[a],provinceandlocation,sourcelocation(town/mountains),tributaryof(river),daminriver,mouth/junctionatlocation(town),mouth/junctioncoordinates,latitude,longitude
0,amandawe river,NaN,"Amandawe, KwaZulu-Natal",north west of Amandawe,north of Scottburgh,NaN,Umpambanyoni River,30°15′30″S 30°45′0″E / 30.25833°S 30.75000°E,-30.25833,30.75000
1,amahlongwa river,NaN,"Amahlongwa, KwaZulu-Natal",south west of Amahlongwa,west of uMkomaas,NaN,Indian Ocean,30°15′30″S 30°44′0″E / 30.25833°S 30.73333°E,-30.25833,30.73333
3,apies river,A2,"Gauteng, Tshwane, Pretoria","Bronberg, southeast of Pretoria","Moretele River, then Crocodile River and Limpo...",Bon Accord Dam,Makapanstad,25°14′24″S 28°08′36″E / 25.24000°S 28.14333°E,-25.24000,28.14333
4,as river (or axel river),C8,Free State,Southeast of Bethlehem,"Liebenbergsvlei River, then Wilge River",Sol Plaatjie Dam,NaN,28°13′27″S 28°21′58″E / 28.22417°S 28.36611°E,-28.22417,28.36611
5,assegaai river,W5,Mpumalanga,North of Wakkerstroom,Mkondo River,Heyshope Dam,Swaziland border,27°04′46″S 31°02′19″E / 27.07944°S 31.03861°E,-27.07944,31.03861


Since this dataset also offers geographical coordinates, we can leverage that information to find nearest natural landmarks. We need to make a large world decision here. We decided any sample locations within 10 km of a river mouth / junction should be representative of the water quality of that river mouth / junction.

How many sample locations are near a river mouth / junction?

In [47]:
def rivers_end(df, THRESH_KM: int, new_col=str) -> pd.DataFrame:
    '''
    input
    -----
    - df: pandas dataframe
    - THRESH_KM: int
    - new_col: str (name of the column that'll be added

    return
    ------
    pandas dataframe with binary column added
    '''
    THRESH_KM = THRESH_KM
    EARTH_RADIUS_KM = 6371

    # river mouth coordinates
    mouth_coords = np.radians(
        df[["latitude", "longitude"]].astype(float).dropna().to_numpy()
    )

    # establish tree, and use the "haversine" distance metric for the geospatial nearest neighbor search
    tree = BallTree(mouth_coords, metric="haversine")

    # sampling point coordinates
    sample_coords = np.radians(
        wq_joined[["latitude", "longitude"]].to_numpy()
    )

    # query the tree for its nearest neighbors
    dist_rad, _ = tree.query(sample_coords, k=1)

    # dist_rad is in the wrong shape
    # print("dist_rad",dist_rad.shape)
    # print("index",_.shape)

    # binary classification
    wq_joined[new_col] = (
        dist_rad[:, 0] * EARTH_RADIUS_KM <= THRESH_KM
    ).astype(int)

    return wq_joined

river_mouthORjunction_df = rivers_end(rivers, THRESH_KM=10, new_col='river_mouthORjunction')
river_mouth_df = rivers_end(river_mouths, THRESH_KM=10, new_col='river_mouth')
river_junction_df = rivers_end(river_junctions, THRESH_KM=10, new_col='river_junction')

### Radians, trees, querys...oh my!!

**The tree query returns distances between mouth coords and sample coords and indices; if the distance in radians, converted to kilometers by multiplying by the radius of the sphere (Earth; 6371 km), is less than or equal to 10 km (away from a river mouth/junction/both), the binary classification would be 1.**

In [50]:

thresh_dict = {}
for i in range(12):
    try:
        thresh_dict[i+1] = rivers_end(river_mouths, THRESH_KM=i+1, new_col='river_mouth')["river_mouth"].value_counts()[1]
    except:
        thresh_dict[i+1] = 0

        
listzipped = list(zip(thresh_dict.keys(), thresh_dict.values()))

pd.DataFrame(listzipped, index=thresh_dict.keys(), columns=['Threshold (kilometers)', 'Number of samples near river mouths']).set_index('Threshold (kilometers)')

,Number of samples near river mouths
Threshold (kilometers),
1,0
2,0
3,0
4,0
5,0
6,0
7,51
8,51
9,51


Logic: As you can see in the table above, the larger the distance threshold, the more sample locations are "near a river mouth". We want to exclude a reasonable amount of sample locations while also maintaining a healthy amount of data to compare with to represent water samples located "near a river mouth".

### Handling Missing Values

It's important for us to know how many attributes we are working with. Below, we can see that there are 22 columns of data to choose from so far, and 9093 rows for each.

Missing values in the dataset were carefully handled to ensure data consistency and granularity. 

In [51]:
# eloquently sort values while observing if there are any missing values
wq_joined = river_junction_df.copy()
wq_joined = wq_joined.dropna(subset='province')
print('shape:',wq_joined.shape)
wq_joined.to_csv('data/wq.csv', index=False)
wq_joined.isna().sum().sort_values(ascending=False)

shape: (9093, 22)


province                         0
country                          0
latitude                         0
longitude                        0
sample date                      0
nir                              0
green                            0
swir16                           0
swir22                           0
ndmi                             0
mndwi                            0
pet                              0
total alkalinity                 0
electrical conductance           0
dissolved reactive phosphorus    0
month                            0
sample_year                      0
pop_density_nn                   0
distance_km_to_pd_cell           0
river_mouthORjunction            0
river_mouth                      0
river_junction                   0
dtype: int64

'NA' was dropped where subset='province' because there were only ~100 'NA' out of over 9000 datapoints leftover. We have a clean dataset without missing values at our disposal for further exploration.

In [52]:
wq_joined.tail()

,province,country,latitude,longitude,sample date,nir,green,swir16,swir22,ndmi,...,total alkalinity,electrical conductance,dissolved reactive phosphorus,month,sample_year,pop_density_nn,distance_km_to_pd_cell,river_mouthORjunction,river_mouth,river_junction
9314,KwaZulu-Natal,South Africa,-27.527500,30.858056,2015-12-23,15296.5,10043.0,16381.0,14443.0,-0.034236,...,38.900,134.0,20.0,2015-12-31,2015,79.402359,0.390899,0,0,0
9315,Mpumalanga,South Africa,-26.861111,28.884722,2015-12-23,15642.5,10294.5,17045.5,14710.0,-0.042921,...,115.800,388.0,20.0,2015-12-31,2015,15.326218,0.251645,0,0,0
9316,North West,South Africa,-26.984722,26.632278,2015-12-23,14945.0,10732.0,18303.0,16281.0,-0.100999,...,104.874,835.0,148.0,2015-12-31,2015,7.830057,0.530196,0,0,0
9317,Free State,South Africa,-27.935000,26.126667,2015-12-23,14727.5,11051.0,18420.0,15724.5,-0.111396,...,128.000,305.0,28.0,2015-12-31,2015,3.002162,0.347046,0,0,0
9318,KwaZulu-Natal,South Africa,-29.744167,29.905833,2015-12-31,15344.5,10461.5,14948.5,12747.0,0.013072,...,37.000,86.4,5.0,2015-12-31,2015,107.789825,0.364569,0,0,0


In [15]:
# raise NotImplementedError()